### Culturomics
#### Method 1: Lexicon Growth & "Dark Matter"
**Replication of J.B. Michel(2011) article  on a Wikipedia Featured-Articles Corpus (2001–2024)**.     
**Course:** Computational Linguistics ,Ca' Foscari University of Venice.     
**Original paper:** Michel, J.-B. et al. (2011). *Quantitative Analysis of Culture Using Millions of Digitized Books*. Science, 331(6014), 176–182.

---

**What this notebook does:**
1. Extracts yearly text snapshots (2001–2024) of approximatly 100 Wikipedia Featured Articles via the MediaWiki Revisions API
2. Builds a diachronic corpus, mirroring Michel's(2011) yearly word-count bins
3. Computes vocabulary size, Type-Token Ratio (TTR), and hapax legomena per year 
4. Estimates "lexical dark matter" the proportion of words absent from WordNet(Dictionary) and the Free Dictionary API

**Methodological note:**
This notebook deliberately uses a *curated sample* (Featured Articles) rather than the full English Wikipedia corpus (6.8M+ articles), due to MediaWiki API rate limits. This is a loose parallel to Michel's (2011) own quality-filtering of 5M books from 15M digitised, but it is not statistically representative of Wikipedia as a whole. The purpose of this analysis is only to check the validity of the methodology.(Explained in Limitations)


### 0. Setup

```bash
pip install requests nltk matplotlib pandas numpy scipy
```


In [1]:
%pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 1.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 1.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install requests


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [34]:
%pip install pandas



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install matplotlib nltk

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 1.4 MB/s eta 0:00:00a 0:00:010m
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 1.4 MB/s eta 0:00:00a 0:00:01
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import time
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import wordnet as wn

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    11,
    'axes.labelsize':    9,
})

PRIMARY   = '#C0392B'
SECONDARY = '#2C3E50'
ACCENT    = '#E67E22'
GREEN     = '#27AE60'

print("Setup complete.")

Setup complete.


[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1020)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1020)>


### 1. Article List 

**Featured Articles:** 100 (underscores instead of spaces, exactly as they appear in the URL).
**Thematic diversity:** science, history, geography,biography, culture, technology (to avoid genre bias to make analyse more representativeness critique).


In [2]:

# FEATURED ARTICLES
ARTICLES = [

    # SCIENCE and KNOWLEDGE
    "Albert_Einstein",
    "Charles_Darwin",
    "Photosynthesis",
    "DNA",
    "Black_hole",
    "Artificial_intelligence",
    "Climate_change",
    "COVID-19_pandemic",
    "Human_genome",
    "Wikipedia",

    # HISTORY and POLITICS
    "World_War_II",
    "French_Revolution",
    "Cold_War",
    "September_11_attacks",
    "Arab_Spring",
    "Occupy_Wall_Street",
    "MeToo_movement",
    "Black_Lives_Matter",
    "WikiLeaks",
    "War_on_terror",

    # TECHNOLOGY: HARDWARE and PLATFORMS 
    "iPhone",
    "Android_(operating_system)",
    "iPod",
    "Tesla,_Inc.",
    "SpaceX",
    "Google",
    "Amazon_(company)",
    "Microsoft",

    # SOCIAL MEDIA and INTERNET 
    "Facebook",
    "YouTube",
    "Twitter",
    "Instagram",
    "Reddit",
    "TikTok",
    "Snapchat",
    "MySpace",
    "Napster",
    "iTunes",
    "Netflix",
    "Spotify",
    "Zoom_(software)",
    "Remote_work",

    # AI and EMERGING TECH
    "ChatGPT",
    "DALL-E",                          # fixed: was DALL·E
    "Virtual_reality",                 # fixed: was Virtual Reality (VR)
    "Augmented_reality",
    "Non-fungible_token",              # fixed: was NFTs
    "Metaverse",
    "Podcast",

    #  FILM and TV 
    "Harry_Potter",
    "The_Lord_of_the_Rings",
    "Shrek_(film)",                    # fixed: was Shrek
    "Avatar_(2009_film)",              # fixed: was Avatar
    "Avengers:_Endgame",
    "Barbie_(film)",                   # fixed: was Barbie
    "Oppenheimer_(film)",              # fixed: was Oppenheimer
    "Frozen_(2013_film)",              # fixed: was Frozen
    "Black_Panther_(film)",            # fixed
    "Joker_(2019_film)",               # fixed
    "Game_of_Thrones",
    "Breaking_Bad",
    "Stranger_Things",
    "Squid_Game",
    "Marvel_Cinematic_Universe",      

    # GAMING 
    "Minecraft",
    "Fortnite",
    "Pokémon_Go",                      # note: accented é is valid in Wikipedia URLs
    "Among_Us",
    "Grand_Theft_Auto_V",
    "The_Last_of_Us",
    "League_of_Legends",
    "World_of_Warcraft",
    "Esports",
    "Roblox",
  

    # MEMES and INTERNET CULTURE 
    "Internet_meme",
    "Rickrolling",
    "Doge_(meme)",                     # fixed: was Doge
    "Pepe_the_Frog",
    "Gangnam_Style",
    "Ice_Bucket_Challenge",
    "Harlem_Shake_(meme)",          

    #  MUSIC and EVENTS 
    "Minecraft",
    "Fortnite",
    "Pokémon_Go",                      # note: accented é is valid in Wikipedia URLs
    "Among_Us",
    "Grand_Theft_Auto_V",
    "The_Last_of_Us",
    "League_of_Legends",
    "World_of_Warcraft",
    "Esports",
    "Roblox",

    # SPORT and CULTURE 
    "2024_Summer_Olympics",            # fixed: was Paris 2024 Olympic Games
    "Pride_parade",

]

print(f"Total articles: {len(ARTICLES)}") 



Total articles: 93


### 2. Configuration

In [3]:
YEAR_START = 2001
YEAR_END   = 2024
YEARS      = list(range(YEAR_START, YEAR_END + 1))

OUTPUT_DIR = Path("culturomics_method1_output")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_FILE = OUTPUT_DIR / "corpus_cache.json"

API_URL = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "CulturomicsEssayBot/1.0 (academic research project)"}

print(f"Period: {YEAR_START}-{YEAR_END} ({len(YEARS)} years)")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Period: 2001-2024 (24 years)
Output folder: /Users/aima/Desktop/Practice/GitHub/research-computational_linguistic/3-notebooks/culturomics_method1_output


### 3. Wikipedia Fetching Functions

These functions retrieve, for each article and year, the revision closest to
mid-January of that year, then strip wikitext markup down to plain text.
If an article didn't exist yet in a given year, it is simply skipped for that year
(this is expected and handled automatically — Wikipedia itself only started in 2001).


In [10]:
def clean_wikitext(raw: str) -> str:
    text = re.sub(r'\[\[[a-z\-]{2,10}:[^\]]+\]\]', ' ', raw)
    text = re.sub(r'\{\{[^{}]*\}\}', ' ', text)
    text = re.sub(r'\{\{[^{}]*\}\}', ' ', text)
    text = re.sub(r'\[\[(?:File|Image|Category):[^\]]*\]\]', ' ', text, flags=re.I)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\[https?://\S+[^\]]*\]', ' ', text)
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'={2,}[^=]+=+', ' ', text)
    text = re.sub(r"'{2,}", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def fetch_yearly_snapshot(article: str, year: int) -> str:
    params_rev = {
        "action": "query", "titles": article, "prop": "revisions",
        "rvprop": "ids|timestamp", "rvlimit": 1, "rvdir": "newer",
        "rvstart": f"{year}-01-01T00:00:00Z",
        "rvend":   f"{year}-03-01T00:00:00Z",
        "format":  "json",
    }
    try:
        r = requests.get(API_URL, params=params_rev, headers=HEADERS, timeout=20)
        if not r.text or len(r.text) < 10:
            time.sleep(30)
            r = requests.get(API_URL, params=params_rev, headers=HEADERS, timeout=20)
        pages = r.json().get("query", {}).get("pages", {})
        page  = list(pages.values())[0]
        if "missing" in page:
            return ""
        revs = page.get("revisions", [])
        if not revs:
            return ""
        rev_id = revs[0]["revid"]
    except Exception as e:
        print(f"    [Step1 error] {article}/{year}: {e}")
        return ""

    params_txt = {
        "action": "query", "revids": rev_id, "prop": "revisions",
        "rvprop": "content", "rvslots": "main", "format": "json",
    }
    try:
        r2 = requests.get(API_URL, params=params_txt, headers=HEADERS, timeout=30)
        if not r2.text or len(r2.text) < 10:
            time.sleep(30)
            r2 = requests.get(API_URL, params=params_txt, headers=HEADERS, timeout=30)
        pages2 = r2.json().get("query", {}).get("pages", {})
        page2  = list(pages2.values())[0]
        raw    = page2["revisions"][0]["slots"]["main"]["*"]
        return clean_wikitext(raw)
    except Exception as e:
        print(f"    [Step2 error] {article}/{year}: {e}")
        return ""

# Проверка
import inspect
src = inspect.getsource(fetch_yearly_snapshot)
print(f"Длина: {len(src)} символов")
print(f"Step2 есть: {'params_txt' in src}")

result = fetch_yearly_snapshot("Albert_Einstein", 2006)
print(f"Тест 2006: {len(result)} символов")


def build_corpus(articles, years, cache_file):
    if cache_file.exists():
        with open(cache_file, encoding='utf-8') as f:
            corpus = json.load(f)
        filled = sum(1 for a in corpus for y in corpus[a] if corpus[a][y])
        print(f"Loaded cache: {filled} non-empty entries already fetched.")
    else:
        corpus = {}

    total = len(articles) * len(years)
    done  = 0   # ← простой счётчик, не пересчитываем весь корпус

    for article in articles:
        if article not in corpus:
            corpus[article] = {}
        for year in years:
            yk = str(year)
            if yk in corpus[article] and corpus[article][yk]:
                done += 1   # ← считаем уже скачанные
                continue
            text = fetch_yearly_snapshot(article, year)
            corpus[article][yk] = text
            if text:
                done += 1   # ← считаем только успешные
            
            # ← ИСПРАВЛЕНО: сохраняем каждые 25 УСПЕШНЫХ (не нулей)
            if done > 0 and done % 25 == 0:
                print(f"  Progress: {done}/{total}  ({done/total*100:.0f}%)")
                with open(cache_file, 'w', encoding='utf-8') as f:
                    json.dump(corpus, f, ensure_ascii=False)
            
            time.sleep(2.0)
        
        time.sleep(5.0)
        print(f"  [done] {article}")

    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False)
    print(f"Done. {done} non-empty snapshots. Cached to: {cache_file}")
    return corpus


def tokenize(text: str) -> list:
    return re.findall(r'\b[a-z]{3,}\b', text.lower())

print("Functions defined OK.")

Длина: 1667 символов
Step2 есть: True
Тест 2006: 38067 символов
Functions defined OK.


In [6]:
# Тест конкретных годов которые падают
import requests, time

for year in [2001, 2003, 2005, 2006, 2007, 2010, 2015, 2020]:
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "titles": "Albert_Einstein", "prop": "revisions",
        "rvprop": "ids|timestamp", "rvlimit": 1, "rvdir": "newer",
        "rvstart": f"{year}-01-01T00:00:00Z",
        "rvend":   f"{year}-03-01T00:00:00Z",
        "format":  "json",
    }, headers=HEADERS, timeout=20)
    
    print(f"{year}: status={r.status_code}, len={len(r.text)}, ok={len(r.text)>50}")
    time.sleep(1)

2001: status=200, len=159, ok=True
2003: status=200, len=290, ok=True
2005: status=200, len=293, ok=True
2006: status=200, len=296, ok=True
2007: status=200, len=296, ok=True
2010: status=200, len=299, ok=True
2015: status=200, len=299, ok=True
2020: status=200, len=299, ok=True


In [8]:
# Точная диагностика 2006 года - оба шага
import requests, json

r = requests.get("https://en.wikipedia.org/w/api.php", params={
    "action": "query", "titles": "Albert_Einstein", "prop": "revisions",
    "rvprop": "ids|timestamp", "rvlimit": 1, "rvdir": "newer",
    "rvstart": "2006-01-01T00:00:00Z",
    "rvend":   "2006-03-01T00:00:00Z",
    "format":  "json",
}, headers=HEADERS, timeout=20)

print(f"Step1 response: {r.text}")

data = r.json()
pages = data.get("query", {}).get("pages", {})
page = list(pages.values())[0]
revs = page.get("revisions", [])
print(f"Revisions: {revs}")

if revs:
    rev_id = revs[0]["revid"]
    print(f"rev_id: {rev_id}")
    
    # Step 2
    r2 = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "revids": rev_id, "prop": "revisions",
        "rvprop": "content", "rvslots": "main", "format": "json",
    }, headers=HEADERS, timeout=30)
    
    print(f"Step2 status: {r2.status_code}, len: {len(r2.text)}")
    print(f"Step2 first 200: {r2.text[:200]}")

Step1 response: {"continue":{"rvcontinue":"20060101081731|33461725","continue":"||"},"query":{"normalized":[{"from":"Albert_Einstein","to":"Albert Einstein"}],"pages":{"736":{"pageid":736,"ns":0,"title":"Albert Einstein","revisions":[{"revid":33458447,"parentid":33420061,"timestamp":"2006-01-01T07:16:46Z"}]}}}}
Revisions: [{'revid': 33458447, 'parentid': 33420061, 'timestamp': '2006-01-01T07:16:46Z'}]
rev_id: 33458447
Step2 status: 200, len: 59036
Step2 first 200: {"batchcomplete":"","query":{"pages":{"736":{"pageid":736,"ns":0,"title":"Albert Einstein","revisions":[{"slots":{"main":{"contentmodel":"wikitext","contentformat":"text/x-wiki","*":"{{Redirect|Einste


In [9]:
import inspect
print(inspect.getsource(fetch_yearly_snapshot))

def fetch_yearly_snapshot(article: str, year: int) -> str:
    params_rev = {
        "action": "query", "titles": article, "prop": "revisions",
        "rvprop": "ids|timestamp", "rvlimit": 1, "rvdir": "newer",
        "rvstart": f"{year}-01-01T00:00:00Z",
        "rvend":   f"{year}-03-01T00:00:00Z",
        "format":  "json",
    }
    try:
        r = requests.get(API_URL, params=params_rev, headers=HEADERS, timeout=20)
        if not r.text or len(r.text) < 10:
            time.sleep(30)
            r = requests.get(API_URL, params=params_rev, headers=HEADERS, timeout=20)
        pages  = r.json().get("query", {}).get("pages", {})
        page   = list(pages.values())[0]
        if "missing" in page:
            return ""
        revs = page.get("revisions", [])
        if not revs:
            return ""
        rev_id = revs[0]["revid"]
    except Exception as e:
        print(f"    [Step1 error] {article}/{year}: {e}")
        return ""

    params_txt = {
        "action": "que

#### 4. Fetch the Corpus

100 articles × 24 years = ~1900 API calls.   
Progress is saved every 25 requests


In [11]:
corpus = build_corpus(ARTICLES, YEARS, CACHE_FILE)

# Quick check
non_empty = sum(1 for a in corpus for y in corpus[a] if corpus[a][y])
print(f"\nNon-empty snapshots: {non_empty} / {len(ARTICLES)*len(YEARS)}")

    [Step2 error] Albert_Einstein/2006: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2007: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2013: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2014: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2015: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2016: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2017: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2018: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2019: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2020: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2021: Expecting value: line 1 column 1 (char 0)
    [Step1 error] Albert_Einstein/2022: Expecting value: line 1 column 1 (char 0)
    [Step1 error

KeyboardInterrupt: 

#### 5. Method 1  Lexicon Growth, TTR, Hapax Legomena

For each year, we pool the text of all articles that existed by that year into a single yearly sub-corpus (this mirrors Michel(2011) yearly word bins, e.g. "98 million words by 1800").    
In other words, the unit of analysis is not a single book or article, but the entire text for a single year. This allows us to construct time series and observe how the use of words, terms, and topics changes over time. This principle directly replicates the methodology of Michel et al., but instead of millions of digitized books, we use historical versions of Wikipedia articles.

We compute:
- **Tokens**:  total word count
- **Types**: unique word count (vocabulary size)
- **TTR**: Type-Token Ratio = types / tokens
- **Hapax legomena**: words occurring exactly once
- **Cumulative vocabulary**:  running total of all unique words ever seen up to that year


In [ ]:
rows = []
cumulative_vocab = set()

for year in YEARS:
    year_tokens = []
    for article in ARTICLES:
        text = corpus.get(article, {}).get(str(year), "")
        if text:
            year_tokens.extend(tokenize(text))

    if not year_tokens:
        continue

    types = set(year_tokens)
    cumulative_vocab.update(types)
    hapax = sum(1 for w in types if year_tokens.count(w) == 1) if len(types) < 5000 else None
    # hapax count skipped for very large yearly vocab to keep runtime reasonable
  

    rows.append({
        "year":          year,
        "tokens":        len(year_tokens),
        "types":         len(types),
        "ttr":           round(len(types) / len(year_tokens), 4),
        "cumulative_vocab": len(cumulative_vocab),
        "n_articles":    sum(1 for a in ARTICLES if corpus.get(a, {}).get(str(year), "")),
    })

df_lex = pd.DataFrame(rows)
df_lex

In [ ]:
# Optimised hapax legomena count (using Counter  much faster than .count() in a loop)
from collections import Counter

hapax_rows = []
for year in YEARS:
    year_tokens = []
    for article in ARTICLES:
        text = corpus.get(article, {}).get(str(year), "")
        if text:
            year_tokens.extend(tokenize(text))
    if not year_tokens:
        continue
    counts = Counter(year_tokens)
    hapax = sum(1 for w, c in counts.items() if c == 1)
    hapax_rows.append({"year": year, "hapax": hapax,
                       "hapax_ratio": round(hapax / len(counts), 4)})

df_hapax = pd.DataFrame(hapax_rows)
df_lex = df_lex.merge(df_hapax, on="year", how="left")
df_lex

## 6. Heaps'/Herdan's Law Validation

Jurafsky & Martin (§2.1) give the empirical relationship between corpus size and
vocabulary size as:

$$|V| = k \cdot N^{\beta}$$

where *N* = total tokens, *|V|* = vocabulary size, and *k, β* are corpus-dependent constants
(typically 0 < β < 1). On a log-log plot, this relationship appears as a straight line.

We fit this here using cumulative tokens vs. cumulative vocabulary across years.


In [ ]:
cum_tokens = np.cumsum(df_lex["tokens"].values)
cum_types  = df_lex["cumulative_vocab"].values

log_N = np.log(cum_tokens)
log_V = np.log(cum_types)

slope, intercept, r_value, p_value, std_err = stats.linregress(log_N, log_V)
beta = slope
k    = np.exp(intercept)

print(f"Heaps' Law fit:  |V| = {k:.3f} · N^{beta:.3f}")
print(f"R² = {r_value**2:.4f}  (closer to 1 = better fit)")
print(f"\nFor comparison, typical English corpora show β between 0.4 and 0.7.")

#### 7. Lexical "Dark Matter" 
**WordNet Coverage**

**Lexical dark matter** Michel(2011) refers to words actively used
in a corpus but absent from standard reference dictionaries.The gap between
the corpus vocabulary and the lexicon documented in authoritative references such as WordNet.

**Methodological limitation ( Limitations section):**
WordNet is a semantic network (150,000 lemmas, organised by synsets), not a historical
lexicographic dictionary like the OED (600,000 entries with etymology) which Michel (2011) used. WordNet's coverage is narrower and more NLP-oriented, so this method likely overestimates the proportion of "dark matter" compared to what a true OED comparison would show. We do not have programmatic access to the OED (subscription-only, no public API), so WordNet is used as the best freely available proxy.


In [ ]:
def in_wordnet(word: str) -> bool:
    return len(wn.synsets(word)) > 0

# Get final-year (2024) vocabulary
final_year_tokens = []
for article in ARTICLES:
    text = corpus.get(article, {}).get(str(YEAR_END), "")
    if text:
        final_year_tokens.extend(tokenize(text))

final_vocab = set(final_year_tokens)
print(f"2024 vocabulary size: {len(final_vocab)} unique words")

in_wn  = sum(1 for w in final_vocab if in_wordnet(w))
out_wn = len(final_vocab) - in_wn
pct_dark = out_wn / len(final_vocab) * 100

print(f"\nIn WordNet:     {in_wn} words ({100-pct_dark:.1f}%)")
print(f"NOT in WordNet: {out_wn} words ({pct_dark:.1f}%)  <- 'lexical dark matter' (WordNet-relative)")

# Sample of "dark matter" words for qualitative inspection
dark_words = [w for w in final_vocab if not in_wordnet(w)]
print(f"\nSample of 20 words not found in WordNet:")
print(sorted(dark_words)[:20])

In [ ]:
# Cross-check with Free Dictionary API (optional, slower — checks a sample only)
# This validates the WordNet-only estimate against a second, independent source,
# mirroring Michel et al.'s two-dictionary approach (OED + Merriam-Webster).

import random

SAMPLE_SIZE = 600  # checking all words would take too long via API; sample instead
sample_dark = random.sample(dark_words, min(SAMPLE_SIZE, len(dark_words)))

def in_free_dictionary(word: str) -> bool:
    try:
        r = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}",
                         timeout=5)
        return r.status_code == 200
    except Exception:
        return False

confirmed_dark = 0
for w in sample_dark:
    if not in_free_dictionary(w):
        confirmed_dark += 1
    time.sleep(0.1)

pct_confirmed = confirmed_dark / len(sample_dark) * 100
print(f"Of {len(sample_dark)} WordNet-dark words sampled:")
print(f"  {confirmed_dark} ({pct_confirmed:.1f}%) ALSO absent from Free Dictionary API")
print(f"  {len(sample_dark)-confirmed_dark} ({100-pct_confirmed:.1f}%) were found in Free Dictionary API")
print(f"\nInterpretation: this {pct_confirmed:.0f}% is a more conservative, cross-validated")
print(f"estimate of true 'dark matter' — words missing from BOTH reference sources.")

## 8. Visualisation — 4-Panel Summary Figure

Mirrors the structure of Michel et al. Figure 2 (panels A and B).


In [ ]:
fig = plt.figure(figsize=(15, 11))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# Panel 1: Vocabulary growth (types per year + cumulative)
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(df_lex["year"], df_lex["types"], color=PRIMARY, alpha=0.5, label="Types/year")
ax1r = ax1.twinx()
ax1r.plot(df_lex["year"], df_lex["cumulative_vocab"], color=SECONDARY, lw=2.5,
          marker='o', ms=3, label="Cumulative vocabulary")
ax1.set_title("A. Vocabulary Growth (2001–2024)", fontweight='bold')
ax1.set_xlabel("Year"); ax1.set_ylabel("Types per year", color=PRIMARY)
ax1r.set_ylabel("Cumulative vocabulary", color=SECONDARY)

# Panel 2: TTR and hapax ratio
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(df_lex["year"], df_lex["ttr"], color=ACCENT, lw=2, marker='s', ms=3, label="TTR")
ax2.plot(df_lex["year"], df_lex["hapax_ratio"], color=GREEN, lw=2, marker='^', ms=3, label="Hapax ratio")
ax2.set_title("B. TTR & Hapax Ratio Over Time", fontweight='bold')
ax2.set_xlabel("Year"); ax2.set_ylabel("Ratio")
ax2.legend(fontsize=8)

# Panel 3: Heaps' Law log-log fit
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(log_N, log_V, color=PRIMARY, s=40, alpha=0.7, zorder=3)
ax3.plot(log_N, intercept + slope*log_N, color=SECONDARY, lw=2, ls='--',
        label=f'β={beta:.3f}, R²={r_value**2:.3f}')
ax3.set_title("C. Heaps' Law: log(Vocabulary) vs log(Tokens)", fontweight='bold')
ax3.set_xlabel("log(cumulative tokens)"); ax3.set_ylabel("log(cumulative types)")
ax3.legend(fontsize=8)

# Panel 4: Dark matter bar chart
ax4 = fig.add_subplot(gs[1, 1])
categories = ['In WordNet', 'NOT in WordNet\n(dark matter)', 'Confirmed dark\n(both sources)']
values = [100-pct_dark, pct_dark, pct_confirmed * pct_dark/100]
colors_bar = [GREEN, ACCENT, PRIMARY]
bars = ax4.bar(categories, values, color=colors_bar, alpha=0.8)
for bar, val in zip(bars, values):
    ax4.text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.1f}%',
             ha='center', fontweight='bold', fontsize=9)
ax4.set_title("D. Lexical 'Dark Matter' (2024 vocabulary)", fontweight='bold')
ax4.set_ylabel("% of vocabulary")
ax4.set_ylim(0, max(values)*1.2)

fig.suptitle("Method 1 — Lexicon Growth on Wikipedia Featured Articles (2001–2024)\n"
             "Replicating Michel et al. (2011) Figure 2",
             fontsize=13, fontweight='bold', y=1.02)

plt.savefig(OUTPUT_DIR / "method1_lexicon_summary.png", dpi=160, bbox_inches='tight')
plt.show()
print("\nFigure saved to:", OUTPUT_DIR / "method1_lexicon_summary.png")

## 9. Key Numbers for the Essay

Run this cell last — it prints a clean summary you can quote directly.


In [ ]:
print("="*60)
print("SUMMARY — METHOD 1 KEY FINDINGS")
print("="*60)
print(f"\nCorpus: {len(ARTICLES)} Wikipedia Featured Articles, {YEAR_START}-{YEAR_END}")
print(f"Final cumulative vocabulary: {df_lex['cumulative_vocab'].iloc[-1]:,} unique words")
print(f"Total tokens processed: {df_lex['tokens'].sum():,}")
print(f"\nTTR range: {df_lex['ttr'].min():.3f} (year {df_lex.loc[df_lex['ttr'].idxmin(),'year']}) "
      f"to {df_lex['ttr'].max():.3f} (year {df_lex.loc[df_lex['ttr'].idxmax(),'year']})")
print(f"\nHeaps' Law: |V| = {k:.2f} · N^{beta:.3f}  (R² = {r_value**2:.3f})")
print(f"\nLexical dark matter (WordNet-relative): {pct_dark:.1f}%")
print(f"Cross-validated dark matter (WordNet + Free Dictionary): ~{pct_confirmed*pct_dark/100:.1f}%")
print(f"\nFor comparison, Michel et al. (2011) found 52% dark matter against OED + Merriam-Webster.")

df_lex.to_csv(OUTPUT_DIR / "method1_results.csv", index=False)
print(f"\nFull results table saved to: {OUTPUT_DIR / 'method1_results.csv'}")